In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [5]:
df = pd.read_csv("Diseases_and_Symptoms_dataset.csv")

In [6]:
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

Rows    : 67933
Columns : 231


In [7]:
display(df.head())

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,sneezing,leg weakness,hysterical behavior,arm lump or mass,bleeding gums,pain in gums,diaper rash,hesitancy,back stiffness or tightness,low urine output
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
print(df.columns.tolist())

['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness', 'palpitations', 'irregular heartbeat', 'breathing fast', 'hoarse voice', 'sore throat', 'difficulty speaking', 'cough', 'nasal congestion', 'throat swelling', 'diminished hearing', 'difficulty in swallowing', 'skin swelling', 'retention of urine', 'leg pain', 'hip pain', 'suprapubic pain', 'blood in stool', 'lack of growth', 'symptoms of the scrotum and testes', 'swelling of scrotum', 'pain in testicles', 'pus draining from ear', 'jaundice', 'white discharge from eye', 'irritable infant', 'abusing alcohol', 'fainting', 'hostile behavior', 'drug abuse', 'sharp abdominal pain', 'feeling ill', 'vomiting', 'headache', 'nausea', 'diarrhea', 'vaginal itching', 'painful urination', 'involuntary urination', 'pain during intercourse', 'frequent urination', 'lower abdominal pain', 'vagina

In [9]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

print(df.columns.tolist())

['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness', 'palpitations', 'irregular heartbeat', 'breathing fast', 'hoarse voice', 'sore throat', 'difficulty speaking', 'cough', 'nasal congestion', 'throat swelling', 'diminished hearing', 'difficulty in swallowing', 'skin swelling', 'retention of urine', 'leg pain', 'hip pain', 'suprapubic pain', 'blood in stool', 'lack of growth', 'symptoms of the scrotum and testes', 'swelling of scrotum', 'pain in testicles', 'pus draining from ear', 'jaundice', 'white discharge from eye', 'irritable infant', 'abusing alcohol', 'fainting', 'hostile behavior', 'drug abuse', 'sharp abdominal pain', 'feeling ill', 'vomiting', 'headache', 'nausea', 'diarrhea', 'vaginal itching', 'painful urination', 'involuntary urination', 'pain during intercourse', 'frequent urination', 'lower abdominal pain', 'vagina

In [10]:
possible_disease_columns = [
    col for col in df.columns
    if col.lower() in ["diseases", "prognosis", "diagnosis", "condition"]
]

print("Possible disease columns:", possible_disease_columns)

Possible disease columns: ['diseases']


In [11]:
target_column = possible_disease_columns[0]

print("Target column:", target_column)

Target column: diseases


In [12]:
X = df.drop(columns=[target_column])
y = df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (67933, 230)
y shape: (67933,)


In [13]:
print("Number of unique diseases:", y.nunique())

print("\nFirst 20 diseases:")
print(y.unique()[:20])

Number of unique diseases: 70

First 20 diseases:
['panic disorder' 'vaginitis' 'problem during pregnancy'
 'acute pancreatitis' 'asthma' 'infectious gastroenteritis'
 'acute sinusitis' 'cornea infection' 'marijuana abuse' 'bursitis'
 'actinic keratosis' 'chronic obstructive pulmonary disease (copd)'
 'spondylosis' 'injury to the arm' 'complex regional pain syndrome'
 'injury to the trunk' 'vulvodynia' 'concussion' 'hypoglycemia'
 'hiatal hernia']


**Check Missing Values**

In [14]:
missing = df.isnull().sum()
print("Columns with missing values:")
display(missing[missing > 0])

Columns with missing values:


,0
seizures,1
delusions or hallucinations,1
excessive urination at night,1
bleeding from eye,1
rectal bleeding,1
...,...
pain in gums,1
diaper rash,1
hesitancy,1
back stiffness or tightness,1


**Check Duplicate**

In [15]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [16]:
unique_values = set()

for column in X.columns:
    values = X[column].dropna().unique()
    unique_values.update(values)

print("Unique values found:")
print(unique_values)

Unique values found:
{np.int64(0), np.int64(1)}


In [17]:
X = X.apply(pd.to_numeric, errors="coerce")

In [18]:
X = X.fillna(0)

In [19]:
print("Remaining missing values:", X.isnull().sum().sum())

Remaining missing values: 0


**Encode disease labels**

In [20]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Original disease:")
print(y.iloc[0])

print("\nEncoded value:")
print(y_encoded[0])

Original disease:
panic disorder

Encoded value:
52


In [21]:
for i, disease in enumerate(label_encoder.classes_[:20]):
    print(i, "→", disease)

0 → actinic keratosis
1 → acute bronchiolitis
2 → acute bronchitis
3 → acute bronchospasm
4 → acute kidney injury
5 → acute pancreatitis
6 → acute sinusitis
7 → allergy
8 → anxiety
9 → arthritis of the hip
10 → asthma
11 → benign prostatic hyperplasia (bph)
12 → bursitis
13 → carpal tunnel syndrome
14 → cholecystitis
15 → chronic obstructive pulmonary disease (copd)
16 → common cold
17 → complex regional pain syndrome
18 → concussion
19 → conjunctivitis due to allergy


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training data :", X_train.shape)
print("Testing data  :", X_test.shape)

Training data : (54346, 230)
Testing data  : (13587, 230)


**Pipeline**

In [30]:
model_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1
        )
    ),

    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )
    )
])

print(model_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(fill_value=-1, strategy='constant')),
                ('classifier',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])


In [31]:
# Create a partially-observed version of training data
rng = np.random.default_rng(42)

X_train_array = X_train.to_numpy(dtype=np.int8).copy()

# Randomly mark 30% of symptoms as UNKNOWN (-1)
mask = rng.random(X_train_array.shape) < 0.30

X_train_masked_array = X_train_array.copy()
X_train_masked_array[mask] = -1

X_train_masked = pd.DataFrame(
    X_train_masked_array,
    columns=X_train.columns,
    index=X_train.index
)

# Combine original + partially observed data
X_train_augmented = pd.concat(
    [X_train, X_train_masked],
    ignore_index=True
)

y_train_augmented = np.concatenate(
    [y_train, y_train]
)

print("Original training data:", X_train.shape)
print("Augmented training data:", X_train_augmented.shape)

Original training data: (54346, 230)
Augmented training data: (108692, 230)


In [32]:
model_pipeline.fit(
    X_train_augmented,
    y_train_augmented
)

print("Model training completed successfully!")

Model training completed successfully!


In [33]:
y_pred = model_pipeline.predict(X_test)

print("Prediction completed!")

Prediction completed!


**Accuracy**

In [34]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 0.8917
Accuracy: 89.17%


**Precision,Recall,F1**

In [35]:
precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Precision: 0.8922
Recall   : 0.8917
F1 Score : 0.8917


In [29]:
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(f"Macro F1: {macro_f1:.4f}")

Macro F1: 0.8916


**Probability Predictions**

In [ ]:
y_proba = model_pipeline.predict_proba(X_test)

print("Probability matrix shape:", y_proba.shape)

Probability matrix shape: (19218, 100)


In [ ]:
print("Number of model classes:", len(model_pipeline.classes_))
print("Number of diseases:", len(label_encoder.classes_))

Number of model classes: 100
Number of diseases: 100


**RF Feature importance**

In [ ]:
rf = model_pipeline.named_steps["classifier"]

feature_importance = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "Symptom": X.columns,
    "Importance": feature_importance
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance_df.head(20).to_string(index=False))

               Symptom  Importance
              headache    0.015290
              vomiting    0.012765
                nausea    0.009956
burning abdominal pain    0.009951
             back pain    0.009763
                 cough    0.009364
      sharp chest pain    0.009249
                 fever    0.008781
              weakness    0.008442
           pelvic pain    0.008194
             dizziness    0.008103
  sharp abdominal pain    0.008005
            joint pain    0.007693
             side pain    0.007619
  lower abdominal pain    0.007039
   shortness of breath    0.007034
     allergic reaction    0.007015
       lower body pain    0.007014
           sore throat    0.006946
    decreased appetite    0.006939


In [ ]:
TOP_K_SYMPTOMS = 50

candidate_symptoms = (
    feature_importance_df
    .head(TOP_K_SYMPTOMS)["Symptom"]
    .tolist()
)

print("Number of candidate symptoms:", len(candidate_symptoms))
print("\nTop candidate symptoms:")

for symptom in candidate_symptoms:
    print("-", symptom)

Number of candidate symptoms: 50

Top candidate symptoms:
- headache
- vomiting
- nausea
- burning abdominal pain
- back pain
- cough
- sharp chest pain
- fever
- weakness
- pelvic pain
- dizziness
- sharp abdominal pain
- joint pain
- side pain
- lower abdominal pain
- shortness of breath
- allergic reaction
- lower body pain
- sore throat
- decreased appetite
- chest tightness
- skin rash
- problems with movement
- ache all over
- difficulty breathing
- chills
- heartburn
- abnormal involuntary movements
- neck pain
- difficulty in swallowing
- shoulder pain
- diarrhea
- ear pain
- nasal congestion
- foot or toe pain
- nosebleed
- knee pain
- leg pain
- vomiting blood
- coryza
- regurgitation.1
- rectal bleeding
- upper abdominal pain
- disturbance of memory
- fluid retention
- abusing alcohol
- symptoms of the face
- hemoptysis
- hot flashes
- arm pain


**Function for TOP-3 Possible Conditions**

In [ ]:
def get_top_predictions(input_data, top_n=3):

    probabilities = model_pipeline.predict_proba(input_data)[0]

    top_indices = np.argsort(probabilities)[-top_n:][::-1]

    results = []

    for index in top_indices:

        disease = label_encoder.inverse_transform([index])[0]
        probability = probabilities[index] * 100

        results.append({
            "Disease": disease,
            "Probability (%)": round(probability, 2)
        })

    return pd.DataFrame(results)

**Disease-symptoms probability table**

In [ ]:
train_probability_data = X_train.copy()

train_probability_data["disease_id"] = y_train

disease_symptom_probability = (
    train_probability_data
    .groupby("disease_id")
    .mean()
)

print(
    "Disease-symptom probability table shape:",
    disease_symptom_probability.shape
)

Disease-symptom probability table shape: (100, 230)


In [ ]:
disease_symptom_probability = (
    disease_symptom_probability
    .reindex(range(len(label_encoder.classes_)))
    .fillna(0)
)

print("Final shape:", disease_symptom_probability.shape)

Final shape: (100, 230)


**Symptom-Input Function**

In [ ]:
def create_symptom_input(
    confirmed_symptoms=None,
    denied_symptoms=None
):

    confirmed_symptoms = confirmed_symptoms or []
    denied_symptoms = denied_symptoms or []

    # -1 = UNKNOWN
    input_data = pd.DataFrame(
        -1,
        index=[0],
        columns=X.columns,
        dtype=np.int8
    )

    # 1 = YES
    for symptom in confirmed_symptoms:
        if symptom in input_data.columns:
            input_data.loc[0, symptom] = 1

    # 0 = NO
    for symptom in denied_symptoms:
        if symptom in input_data.columns:
            input_data.loc[0, symptom] = 0

    return input_data

**Shannon Entropy**

In [ ]:
def calculate_entropy(probabilities):

    probabilities = np.asarray(probabilities)

    # Remove zero probabilities
    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return entropy

In [ ]:
example_probabilities = np.array([
    0.40,
    0.35,
    0.25
])

entropy = calculate_entropy(example_probabilities)

print("Entropy:", entropy)

Entropy: 1.55887184844536


**Calculate Next best question**

In [ ]:
def find_next_best_question(
    current_input,
    answered_symptoms,
    top_k=50
):

    # --------------------------------
    # Current disease probabilities
    # --------------------------------
    current_probabilities = (
        model_pipeline
        .predict_proba(current_input)[0]
    )

    # --------------------------------
    # Current uncertainty
    # --------------------------------
    current_entropy = calculate_entropy(
        current_probabilities
    )

    # --------------------------------
    # Select top important symptoms
    # --------------------------------
    important_symptoms = (
        feature_importance_df
        .head(top_k)["Symptom"]
        .tolist()
    )

    # --------------------------------
    # Remove already answered symptoms
    # --------------------------------
    unknown_symptoms = [
        symptom
        for symptom in important_symptoms
        if symptom not in answered_symptoms
    ]

    # If no symptoms are left
    if not unknown_symptoms:
        return pd.DataFrame()

    results = []

    # --------------------------------
    # Evaluate each unknown symptom
    # --------------------------------
    for symptom in unknown_symptoms:

        # ==============================
        # Simulate YES
        # ==============================
        yes_input = current_input.copy()
        yes_input.loc[0, symptom] = 1

        yes_probabilities = (
            model_pipeline
            .predict_proba(yes_input)[0]
        )

        yes_entropy = calculate_entropy(
            yes_probabilities
        )

        # ==============================
        # Simulate NO
        # ==============================
        no_input = current_input.copy()
        no_input.loc[0, symptom] = 0

        no_probabilities = (
            model_pipeline
            .predict_proba(no_input)[0]
        )

        no_entropy = calculate_entropy(
            no_probabilities
        )

        # ==============================
        # Estimate P(YES)
        # ==============================
        symptom_probabilities = (
            disease_symptom_probability[symptom]
            .values
        )

        p_yes = np.sum(
            current_probabilities *
            symptom_probabilities
        )

        p_yes = np.clip(
            p_yes,
            0,
            1
        )

        p_no = 1 - p_yes

        # ==============================
        # Expected Entropy
        # ==============================
        expected_entropy = (
            p_yes * yes_entropy
            +
            p_no * no_entropy
        )

        # ==============================
        # Information Gain
        # ==============================
        information_gain = (
            current_entropy
            -
            expected_entropy
        )

        results.append({
            "Symptom": symptom,
            "P(Yes)": p_yes,
            "P(No)": p_no,
            "Current Entropy": current_entropy,
            "Expected Entropy": expected_entropy,
            "Information Gain": information_gain
        })

    # --------------------------------
    # Sort by highest information gain
    # --------------------------------
    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        by="Information Gain",
        ascending=False
    ).reset_index(drop=True)

    return results_df

**Display the best question**

In [ ]:
def adaptive_disease_prediction(
    initial_symptoms,
    confidence_threshold=0.80,
    max_questions=7
):

    confirmed_symptoms = set(initial_symptoms)
    denied_symptoms = set()
    answered_symptoms = set(initial_symptoms)

    question_count = 0

    current_input = create_symptom_input(
        confirmed_symptoms=confirmed_symptoms,
        denied_symptoms=denied_symptoms
    )

    print("=" * 60)
    print("MEDIGUIDE AI")
    print("ADAPTIVE SYMPTOM ASSESSMENT")
    print("=" * 60)

    print("\nInitial Symptoms:")

    for symptom in confirmed_symptoms:
        print("✓", symptom)

    while question_count < max_questions:

        # --------------------------------
        # Current disease probabilities
        # --------------------------------
        probabilities = model_pipeline.predict_proba(
            current_input
        )[0]

        # Highest probability disease
        top_index = np.argmax(probabilities)

        top_disease = label_encoder.inverse_transform(
            [top_index]
        )[0]

        top_probability = probabilities[top_index]

        print("\n" + "-" * 60)

        print(
            f"Current Prediction: {top_disease}"
        )

        print(
            f"Confidence: {top_probability * 100:.2f}%"
        )

        # --------------------------------
        # Stopping condition
        # --------------------------------
        if top_probability >= confidence_threshold:

            print("\nConfidence threshold reached.")
            break

        # --------------------------------
        # Find next best question
        # --------------------------------
        question_results = find_next_best_question(
            current_input,
            answered_symptoms,
            top_k=50
        )

        if question_results.empty:

            print(
                "\nNo unanswered candidate symptoms remaining."
            )
            break

        best_question = question_results.iloc[0]

        symptom = best_question["Symptom"]

        print("\nNext Best Question:")

        print(
            f"Do you have '{symptom}'?"
        )

        print(
            "Information Gain:",
            round(
                best_question["Information Gain"],
                4
            )
        )

        # --------------------------------
        # Get user answer
        # --------------------------------
        answer = input(
            "Enter yes/no: "
        ).strip().lower()

        while answer not in ["yes", "no", "y", "n"]:

            print("Please enter yes or no.")

            answer = input(
                "Enter yes/no: "
            ).strip().lower()

        # --------------------------------
        # Update symptom state
        # --------------------------------
        if answer in ["yes", "y"]:

            confirmed_symptoms.add(symptom)

            current_input.loc[
                0, symptom
            ] = 1

            print(
                f"✓ Confirmed: {symptom}"
            )

        else:

            denied_symptoms.add(symptom)

            current_input.loc[
                0, symptom
            ] = 0

            print(
                f"✗ Denied: {symptom}"
            )

        answered_symptoms.add(symptom)

        question_count += 1

    # --------------------------------
    # FINAL PREDICTION
    # --------------------------------

    final_probabilities = (
        model_pipeline.predict_proba(
            current_input
        )[0]
    )

    final_index = np.argmax(
        final_probabilities
    )

    final_disease = label_encoder.inverse_transform(
        [final_index]
    )[0]

    final_confidence = (
        final_probabilities[final_index]
    )

    print("\n" + "=" * 60)
    print("FINAL RESULT")
    print("=" * 60)

    # --------------------------------
    # Confidence >= 80%
    # --------------------------------
    if final_confidence >= confidence_threshold:

        print(
            "Predicted Disease:",
            final_disease
        )

        print(
            f"Confidence: "
            f"{final_confidence * 100:.2f}%"
        )

    # --------------------------------
    # Confidence < 80%
    # --------------------------------
    else:

        print(
            f"Confidence is below "
            f"{confidence_threshold * 100:.0f}%."
        )

        print("\nTop 3 Possible Conditions:")

        top_3_indices = np.argsort(
            final_probabilities
        )[-3:][::-1]

        for rank, index in enumerate(
            top_3_indices,
            start=1
        ):

            disease = label_encoder.inverse_transform(
                [index]
            )[0]

            probability = (
                final_probabilities[index] * 100
            )

            print(
                f"{rank}. {disease} "
                f"→ {probability:.2f}%"
            )

    print(
        f"\nQuestions Asked: "
        f"{question_count}"
    )

    return {
        "disease": final_disease,
        "confidence": final_confidence,
        "questions_asked": question_count,
        "confirmed_symptoms": list(
            confirmed_symptoms
        ),
        "denied_symptoms": list(
            denied_symptoms
        )
    }

In [ ]:
# Show available symptoms
print("Please enter your starting symptoms.")
print("Use the symptom names exactly as they appear in the dataset.\n")

user_input = input(
    "Enter symptoms separated by commas: "
)

# Convert user input into a list
initial_symptoms = [
    symptom.strip().lower()
    for symptom in user_input.split(",")
    if symptom.strip()
]

# Validate symptoms
valid_symptoms = []
invalid_symptoms = []

for symptom in initial_symptoms:

    if symptom in X.columns:
        valid_symptoms.append(symptom)
    else:
        invalid_symptoms.append(symptom)

# Display invalid symptoms
if invalid_symptoms:
    print("\nThese symptoms were not found in the dataset:")

    for symptom in invalid_symptoms:
        print("-", symptom)

# Check whether at least one valid symptom was entered
if not valid_symptoms:
    print("\nNo valid symptoms entered.")

else:
    print("\nStarting symptoms:")

    for symptom in valid_symptoms:
        print("✓", symptom)

    result = adaptive_disease_prediction(
        initial_symptoms=valid_symptoms,
        confidence_threshold=0.80,
        max_questions=7
    )

Please enter your starting symptoms.
Use the symptom names exactly as they appear in the dataset.

Enter symptoms separated by commas: fever

Starting symptoms:
✓ fever
MEDIGUIDE AI
ADAPTIVE SYMPTOM ASSESSMENT

Initial Symptoms:
✓ fever

------------------------------------------------------------
Current Prediction: otitis media
Confidence: 9.00%

Next Best Question:
Do you have 'chills'?
Information Gain: 0.3375
Enter yes/no: no
✗ Denied: chills

------------------------------------------------------------
Current Prediction: otitis media
Confidence: 9.00%

Next Best Question:
Do you have 'difficulty breathing'?
Information Gain: 0.2788
Enter yes/no: yes
✓ Confirmed: difficulty breathing

------------------------------------------------------------
Current Prediction: nose disorder
Confidence: 23.00%

Next Best Question:
Do you have 'nosebleed'?
Information Gain: 0.3713
Enter yes/no: no
✗ Denied: nosebleed

------------------------------------------------------------
Current Predicti